# Run the Document VQA project on Colab
A GPU is recommended, but the demo can also run on CPU. Upload Document_VQA_Project.zip to /content/. This notebook does not retrain the model.


**Translation note:** This notebook is an English translation of the supplied notebook. Saved output messages were translated for readability; numerical results were preserved. Translation does not represent a new training or evaluation run.


In [1]:
from pathlib import Path
import zipfile
from google.colab import drive
drive.mount('/content/drive')
zip_path = Path('/content/Document_VQA_Project.zip')
assert zip_path.exists(), 'Upload the project ZIP to /content first.'
with zipfile.ZipFile(zip_path) as z:
    z.extractall('/content/vqa_code')
CODE_DIR = Path('/content/vqa_code/Document_VQA_Project')
print(CODE_DIR)


Mounted at /content/drive
/content/vqa_code/Document_VQA_Project


In [2]:
%pip install -q -r /content/vqa_code/Document_VQA_Project/requirements.txt
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-eng


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 10.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


If a restart is required, or libraries were imported before installation, select Runtime → Restart session, then run the cell below. Do not rerun training. This demo does not use Gradio or Diffusers.


In [3]:
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
CODE_DIR = Path('/content/vqa_code/Document_VQA_Project')
PROJECT_DATA_DIR = Path('/content/drive/MyDrive/Document_VQA_Project')
MODEL_DIR = PROJECT_DATA_DIR / 'checkpoints/donut_decoder_finetuned_clean'
assert (MODEL_DIR / 'config.json').exists(), 'Check the checkpoint path on Drive.'
assert list(MODEL_DIR.glob('*.safetensors')) or (MODEL_DIR / 'pytorch_model.bin').exists(), 'Missing model weights.'
print('Checkpoint found:', MODEL_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint found: /content/drive/MyDrive/Document_VQA_Project/checkpoints/donut_decoder_finetuned_clean


In [4]:
import subprocess, sys
# Run the demo in a separate process to avoid old libraries in the kernel.
# Stop the old server before rerunning this cell.
if 'demo_process' in globals() and demo_process.poll() is None:
    demo_process.terminate()
    demo_process.wait(timeout=15)
demo_log = open('/content/vqa_demo.log', 'w')
demo_process = subprocess.Popen(
    [sys.executable, 'app.py', '--model-path', str(MODEL_DIR),
     '--host', '0.0.0.0', '--port', '5000'],
    cwd=CODE_DIR, stdout=demo_log, stderr=subprocess.STDOUT,
)
print('Demo started. Log: /content/vqa_demo.log')


Demo started. Log: /content/vqa_demo.log


In [5]:
import time, urllib.request
from google.colab import output
ready = False
for attempt in range(30):
    if demo_process.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:5000/', timeout=2) as r:
            ready = r.status == 200
        if ready:
            break
    except OSError:
        time.sleep(1)
if ready:
    output.serve_kernel_port_as_iframe(5000, height=850)
else:
    print(Path('/content/vqa_demo.log').read_text()[-5000:])


<IPython.core.display.Javascript object>

Select an image from data_clean/images/ or data_clean/receipts/images/ and ask a question with a known answer.

In [ ]:
# Optional: rescore saved raw JSON files without running the models.
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'src.summarize_results',
     '--results-dir', str(PROJECT_DATA_DIR / 'results')],
    cwd=CODE_DIR, check=True,
)
